In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/biohub-cell-tracking-during-development/sample_submission.csv
/kaggle/input/competitions/biohub-cell-tracking-during-development/test/44b6_0b24845f.zarr/zarr.json
/kaggle/input/competitions/biohub-cell-tracking-during-development/test/44b6_0b24845f.zarr/0/zarr.json
/kaggle/input/competitions/biohub-cell-tracking-during-development/test/44b6_0b24845f.zarr/0/c/7/0/0/0
/kaggle/input/competitions/biohub-cell-tracking-during-development/test/44b6_0b24845f.zarr/0/c/47/0/0/0
/kaggle/input/competitions/biohub-cell-tracking-during-development/test/44b6_0b24845f.zarr/0/c/17/0/0/0
/kaggle/input/competitions/biohub-cell-tracking-during-development/test/44b6_0b24845f.zarr/0/c/81/0/0/0
/kaggle/input/competitions/biohub-cell-tracking-during-development/test/44b6_0b24845f.zarr/0/c/19/0/0/0
/kaggle/input/competitions/biohub-cell-tracking-during-development/test/44b6_0b24845f.zarr/0/c/22/0/0/0
/kaggle/input/competitions/biohub-cell-tracking-during-development/test/44b6_0b24845

# Biohub – Cell Tracking During Development

## Chapter 9: Building a Candidate Filter for LoG Detections

**Objective:** Train a simple model to rank LoG blob detections by how likely they are to correspond to ground-truth cell centers.

# 🎯 Learning Objectives

By the end of this notebook, I will be able to:

- turn blob detections into a labeled training table
- engineer spatial and local-context features for each candidate detection
- train a simple classifier to predict whether a blob is GT-like
- rank LoG detections by predicted quality
- evaluate whether filtering candidates improves precision while preserving useful recall

# 🧠 Background Theory

In the previous chapters, I learned that LoG blob detection can recover more labeled centroids than local maxima, but it also produces many extra detections.

That suggests a two-stage strategy:

1. **Generate candidates** using a permissive detector such as LoG
2. **Filter or rank candidates** using learned features

In this notebook, I will treat each LoG detection as a training example and predict whether it is close to a ground-truth centroid.

# 🗺️ Chapter Roadmap

This notebook will proceed in seven stages:

1. **Reload the sample and labels**
2. **Recreate LoG blob detections**
3. **Label detections as matched or unmatched**
4. **Engineer candidate-level features**
5. **Train a simple candidate classifier**
6. **Rank detections by predicted GT-likeness**
7. **Evaluate whether filtering improves the candidate set**

# 📚 New Vocabulary

## Candidate Filter
A model that decides which candidate detections are worth keeping.

## Ranking Score
A predicted score that orders detections from most GT-like to least GT-like.

## Center of Mass
The intensity-weighted average location of a patch.

## GT-like
A detection whose appearance and context resemble detections that match the sparse ground-truth labels.

In [2]:
!pip install zarr -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.7/363.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 42.9 MB/s eta 0:00:00


In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import zarr

from skimage.feature import blob_log
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier

In [4]:
print("Imports successful.")

Imports successful.


# Step 1 — Reload the Same Sample and Timepoint

To make this chapter consistent with the previous analysis, I will continue using the same training sample and the same labeled timepoint.

In [5]:
competition_path = Path("/kaggle/input/competitions/biohub-cell-tracking-during-development")
train_path = competition_path / "train"

sample_id = "44b6_12dfb391"
zarr_path = train_path / f"{sample_id}.zarr"
geff_path = train_path / f"{sample_id}.geff"

# Load image
volume = zarr.open(zarr_path, mode="r")
image = volume["0"]

# Load graph
geff = zarr.open(geff_path, mode="r")

node_ids = geff["nodes/ids"][:]
t_vals = geff["nodes/props/t/values"][:]
z_vals = geff["nodes/props/z/values"][:]
y_vals = geff["nodes/props/y/values"][:]
x_vals = geff["nodes/props/x/values"][:]

nodes_df = pd.DataFrame({
    "node_id": node_ids,
    "t": t_vals,
    "z": z_vals,
    "y": y_vals,
    "x": x_vals
})

t = 8
frame_3d = image[t]
gt_nodes_t = nodes_df[nodes_df["t"] == t].copy().sort_values("z")
gt_points = gt_nodes_t[["z", "y", "x"]].to_numpy()

print("Frame shape:", frame_3d.shape)
print("Ground-truth nodes:", len(gt_nodes_t))
gt_nodes_t

Frame shape: (64, 256, 256)
Ground-truth nodes: 11


,node_id,t,z,y,x
82,114000000041,8,10,237,65
81,114000000040,8,15,184,60
87,114000000080,8,17,248,101
90,114000000084,8,18,2,116
89,114000000082,8,22,199,63
84,114000000043,8,25,208,115
83,114000000042,8,27,156,0
88,114000000081,8,28,122,100
80,114000000037,8,37,50,192
86,114000000063,8,40,198,98


# Step 2 — Recreate LoG Candidate Detections

I will reuse the strict LoG detector from the previous notebook and then label each candidate as matched or unmatched relative to the ground-truth centroids.

In [6]:
def detect_blobs_log_3d_slicewise(
    frame_3d,
    min_sigma=2.5,
    max_sigma=3.5,
    num_sigma=3,
    threshold=0.15
):
    detections = []

    for z in range(frame_3d.shape[0]):
        slice_2d = frame_3d[z].astype(np.float32)

        s_min, s_max = slice_2d.min(), slice_2d.max()
        if s_max > s_min:
            slice_norm = (slice_2d - s_min) / (s_max - s_min)
        else:
            slice_norm = slice_2d

        blobs = blob_log(
            slice_norm,
            min_sigma=min_sigma,
            max_sigma=max_sigma,
            num_sigma=num_sigma,
            threshold=threshold
        )

        for blob in blobs:
            y, x, sigma = blob
            detections.append({
                "z": int(z),
                "y": int(round(y)),
                "x": int(round(x)),
                "sigma": float(sigma)
            })

    return pd.DataFrame(detections)

In [7]:
def pairwise_distances_3d(gt_points, pred_points):
    gt = gt_points[:, None, :]
    pred = pred_points[None, :, :]
    distances = np.sqrt(((gt - pred) ** 2).sum(axis=2))
    return distances

In [8]:
blob_df = detect_blobs_log_3d_slicewise(frame_3d)

blob_points = blob_df[["z", "y", "x"]].to_numpy()
dist_blob = pairwise_distances_3d(gt_points, blob_points)

min_dist_pred = dist_blob.min(axis=0)
match_radius = 6

blob_df["nearest_gt_dist"] = min_dist_pred
blob_df["matched"] = blob_df["nearest_gt_dist"] <= match_radius

print("Blob detections:", len(blob_df))
print(blob_df["matched"].value_counts())
blob_df.head()

Blob detections: 1478
matched
False    1453
True       25
Name: count, dtype: int64


,z,y,x,sigma,nearest_gt_dist,matched
0,0,92,0,3.5,69.462220,False
1,0,183,6,3.5,38.652296,False
2,0,118,19,3.5,50.338852,False
3,0,212,75,3.5,28.231188,False
4,0,208,19,3.5,49.819675,False


# Step 3 — Engineer Candidate Features

I will extract both 2D patch features and simple 3D support features for each LoG detection.

The goal is to describe not just brightness, but also centering, local structure, and consistency across neighboring z-slices.

In [9]:
def extract_candidate_features(frame_3d, detections_df, patch_radius=4):
    rows = []
    z_max, y_max, x_max = frame_3d.shape

    for _, row in detections_df.iterrows():
        z = int(row["z"])
        y = int(row["y"])
        x = int(row["x"])

        slice_2d = frame_3d[z].astype(np.float32)

        # ----- 2D patch -----
        y0 = max(0, y - patch_radius)
        y1 = min(y_max, y + patch_radius + 1)
        x0 = max(0, x - patch_radius)
        x1 = min(x_max, x + patch_radius + 1)

        patch = slice_2d[y0:y1, x0:x1]

        center_intensity = float(slice_2d[y, x])
        patch_mean = float(patch.mean())
        patch_max = float(patch.max())
        patch_std = float(patch.std())
        local_contrast = center_intensity - patch_mean

        # brightest pixel location in patch
        max_idx = np.unravel_index(np.argmax(patch), patch.shape)
        max_y_patch, max_x_patch = max_idx

        patch_center_y = y - y0
        patch_center_x = x - x0

        center_to_patch_max_dist = float(
            np.sqrt((max_y_patch - patch_center_y)**2 + (max_x_patch - patch_center_x)**2)
        )

        # center of mass (intensity-weighted)
        yy, xx = np.indices(patch.shape)
        patch_sum = patch.sum()

        if patch_sum > 0:
            com_y = float((yy * patch).sum() / patch_sum)
            com_x = float((xx * patch).sum() / patch_sum)
            patch_com_offset = float(
                np.sqrt((com_y - patch_center_y)**2 + (com_x - patch_center_x)**2)
            )
        else:
            patch_com_offset = np.nan

        # border feature
        border_margin = min(y, x, y_max - 1 - y, x_max - 1 - x)
        is_near_border = int(border_margin < patch_radius)

        # ----- z-support -----
        z_vals_local = []
        for zz in [z - 1, z, z + 1]:
            if 0 <= zz < z_max:
                z_vals_local.append(float(frame_3d[zz, y, x]))

        z_center_intensity = float(frame_3d[z, y, x])
        z_support_mean = float(np.mean(z_vals_local))
        z_support_std = float(np.std(z_vals_local))
        z_support_min = float(np.min(z_vals_local))
        z_support_max = float(np.max(z_vals_local))

        rows.append({
            "z": z,
            "y": y,
            "x": x,
            "sigma": float(row["sigma"]),
            "nearest_gt_dist": float(row["nearest_gt_dist"]),
            "matched": bool(row["matched"]),

            "center_intensity": center_intensity,
            "patch_mean": patch_mean,
            "patch_max": patch_max,
            "patch_std": patch_std,
            "local_contrast": local_contrast,

            "center_to_patch_max_dist": center_to_patch_max_dist,
            "patch_com_offset": patch_com_offset,
            "is_near_border": is_near_border,

            "z_center_intensity": z_center_intensity,
            "z_support_mean": z_support_mean,
            "z_support_std": z_support_std,
            "z_support_min": z_support_min,
            "z_support_max": z_support_max,
        })

    return pd.DataFrame(rows)

In [10]:
candidate_df = extract_candidate_features(frame_3d, blob_df, patch_radius=4)

print("Candidate rows:", len(candidate_df))
candidate_df.head()

Candidate rows: 1478


,z,y,x,sigma,nearest_gt_dist,matched,center_intensity,patch_mean,patch_max,patch_std,local_contrast,center_to_patch_max_dist,patch_com_offset,is_near_border,z_center_intensity,z_support_mean,z_support_std,z_support_min,z_support_max
0,0,92,0,3.5,69.462220,False,1280.0,907.822205,1280.0,174.333572,372.177795,0.000000,1.851372,1,1280.0,1438.5,158.5,1280.0,1597.0
1,0,183,6,3.5,38.652296,False,1250.0,1016.234558,1286.0,158.421249,233.765442,1.000000,0.162069,0,1250.0,1121.5,128.5,993.0,1250.0
2,0,118,19,3.5,50.338852,False,867.0,742.172852,920.0,97.855751,124.827148,1.414214,0.161427,0,867.0,978.5,111.5,867.0,1090.0
3,0,212,75,3.5,28.231188,False,1028.0,864.765442,1050.0,113.143227,163.234558,1.000000,0.166543,0,1028.0,891.0,137.0,754.0,1028.0
4,0,208,19,3.5,49.819675,False,885.0,822.703674,928.0,76.585213,62.296326,5.656854,0.130084,0,885.0,895.0,10.0,885.0,905.0


# Step 4 — Build a Training Table

Each LoG detection will now become a supervised training example:

- **target = matched**
- **features = patch + spatial + z-support features**

In [11]:
feature_cols = [
    "sigma",
    "center_intensity",
    "patch_mean",
    "patch_max",
    "patch_std",
    "local_contrast",
    "center_to_patch_max_dist",
    "patch_com_offset",
    "is_near_border",
    "z_center_intensity",
    "z_support_mean",
    "z_support_std",
    "z_support_min",
    "z_support_max",
]

model_df = candidate_df[feature_cols + ["matched"]].copy()

X = model_df[feature_cols]
y = model_df["matched"].astype(int)

print("X shape:", X.shape)
print("Positive class count:", y.sum())
print("Negative class count:", len(y) - y.sum())

X shape: (1478, 14)
Positive class count: 25
Negative class count: 1453


# Step 5 — Train a Simple Candidate Classifier

I will use a class-balanced Random Forest to predict whether a blob detection is GT-like.

This is not meant to be the final competition model. It is a first test of whether engineered candidate features can help rank detections better than raw LoG alone.

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))
print("Train positives:", y_train.sum())
print("Test positives:", y_test.sum())

Train size: 1108
Test size: 370
Train positives: 19
Test positives: 6


In [13]:
clf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    max_depth=6
)

clf.fit(X_train, y_train)

print("Training complete.")

Training complete.


In [14]:
importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": clf.feature_importances_
}).sort_values("importance", ascending=False)

importance_df

,feature,importance
7,patch_com_offset,0.203110
13,z_support_max,0.126799
10,z_support_mean,0.100062
11,z_support_std,0.077443
2,patch_mean,0.074521
1,center_intensity,0.074006
9,z_center_intensity,0.071533
5,local_contrast,0.064255
12,z_support_min,0.061909
3,patch_max,0.057183


# Step 6 — Rank All LoG Candidates by GT-Likeness

Now I will score every LoG candidate using the trained model and sort detections from most GT-like to least GT-like.

In [15]:
candidate_df = candidate_df.copy()
candidate_df["gt_like_score"] = clf.predict_proba(candidate_df[feature_cols])[:, 1]

ranked_df = candidate_df.sort_values("gt_like_score", ascending=False)
ranked_df.head(20)

,z,y,x,sigma,nearest_gt_dist,matched,center_intensity,patch_mean,patch_max,patch_std,local_contrast,center_to_patch_max_dist,patch_com_offset,is_near_border,z_center_intensity,z_support_mean,z_support_std,z_support_min,z_support_max,gt_like_score
274,17,245,99,3.5,3.605551,True,1493.0,1213.061768,1493.0,141.490509,279.938232,0.000000,0.046691,0,1493.0,1348.666667,117.173186,1206.0,1493.0,0.705798
94,8,234,64,3.5,3.741657,True,1437.0,1152.419800,1470.0,185.886795,284.580200,1.000000,0.113143,0,1437.0,1362.000000,92.285788,1232.0,1437.0,0.701110
523,26,209,118,3.5,3.316625,True,1463.0,1296.814819,1523.0,116.985077,166.185181,4.123106,0.118907,0,1463.0,1370.666667,127.761062,1190.0,1463.0,0.691575
251,16,183,57,3.5,3.316625,True,1353.0,1194.716064,1369.0,111.440102,158.283936,1.414214,0.099648,0,1353.0,1289.000000,73.543638,1186.0,1353.0,0.688853
71,7,234,64,3.5,4.358899,True,1417.0,1121.481445,1418.0,164.917633,295.518555,1.414214,0.088469,0,1417.0,1317.333333,155.306865,1098.0,1437.0,0.682034
339,21,197,62,3.5,2.449490,True,1809.0,1581.580200,1834.0,162.448914,227.419800,1.000000,0.020043,0,1809.0,1796.000000,135.442485,1624.0,1955.0,0.676028
250,16,245,99,3.5,3.741657,True,1347.0,1151.604980,1347.0,126.122292,195.395020,0.000000,0.077463,0,1347.0,1289.000000,194.614148,1027.0,1493.0,0.664900
1151,45,117,143,3.5,5.916080,True,1869.0,1600.827148,1943.0,204.413101,268.172852,2.828427,0.107938,0,1869.0,1873.333333,136.797498,1708.0,2043.0,0.653777
1133,44,120,143,3.5,5.744563,True,1911.0,1615.876587,1911.0,167.486008,295.123413,0.000000,0.036428,0,1911.0,1939.000000,249.003347,1649.0,2257.0,0.646166
137,10,240,67,3.5,3.605551,True,1294.0,1105.000000,1294.0,136.249771,189.000000,0.000000,0.164304,0,1294.0,1200.666667,68.397531,1132.0,1294.0,0.641773


In [16]:
top_k = 50
top_ranked = ranked_df.head(top_k)

print("Top-k candidate count:", len(top_ranked))
print("Matched detections inside top-k:", top_ranked["matched"].sum())
print("Match rate inside top-k:", top_ranked["matched"].mean())

Top-k candidate count: 50
Matched detections inside top-k: 20
Match rate inside top-k: 0.4


# 📊 Results

The candidate filter substantially improved the quality of the LoG candidate ranking.

Raw LoG detections:

- **1478 total candidates**
- **25 matched candidates**
- **match rate:** 25 / 1478 ≈ 0.017

Top 50 ranked candidates:

- **50 candidates**
- **20 matched candidates**
- **match rate:** 0.40

This means the learned candidate filter made the top-ranked pool far more concentrated with GT-like detections.

# 🔍 Analysis

The most important feature was `patch_com_offset`, suggesting that GT-like detections tend to have a more centered intensity mass within the patch.

The next strongest features were `z_support_max`, `z_support_mean`, and `z_support_std`, which suggests that 3D context matters. A candidate is more likely to be GT-like when its location has coherent support across neighboring z-slices.

Raw brightness alone was not the main signal. Spatial coherence, centering, and 3D context appear more useful for filtering noisy blob detections.

# 🚀 Next Steps

## Chapter 10 — Build a Filtered Detector

Next, I want to convert the ranked candidate list into an actual filtered detector.

Goals:

- keep only the top N candidates
- score top 25, top 50, top 100, and top 200
- compare recall and precision-like behavior
- visualize the filtered detections
- choose a first practical detector configuration